In [17]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [18]:
df = pd.read_csv("covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [19]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["has_covid"]), df["has_covid"], test_size=0.2, random_state=42)

## Without column transformer

In [21]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.fit_transform(X_test[["fever"]])

In [22]:
oe = OrdinalEncoder()
X_train_cough = oe.fit_transform(X_train[["cough"]])
X_test_cough = oe.fit_transform(X_test[["cough"]])

In [23]:
ohe = OneHotEncoder(drop='first')
X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]]).toarray()
X_test_gender_city = ohe.fit_transform(X_test[["gender", "city"]]).toarray()

In [24]:
X_train_age = X_train.drop(columns=["gender", "fever", "cough", "city"]).values
X_test_age = X_test.drop(columns=["gender", "fever", "cough", "city"]).values
X_train_age.shape

(80, 1)

In [25]:
X_train_tranformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_tranformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)
X_train_tranformed

array([[ 81., 101.,   0.,   0.,   0.,   1.,   0.],
       [  5., 100.,   0.,   0.,   1.,   0.,   0.],
       [ 19., 100.,   0.,   0.,   1.,   0.,   0.],
       [ 27., 100.,   1.,   1.,   0.,   0.,   0.],
       [ 73., 103.,   0.,   1.,   0.,   0.,   0.],
       [ 70., 103.,   1.,   0.,   1.,   0.,   1.],
       [ 49., 102.,   0.,   1.,   0.,   0.,   0.],
       [ 51., 101.,   0.,   0.,   1.,   0.,   1.],
       [ 64., 101.,   0.,   1.,   0.,   0.,   0.],
       [ 83., 101.,   0.,   0.,   1.,   0.,   0.],
       [ 65.,  98.,   0.,   0.,   0.,   1.,   0.],
       [ 18., 104.,   0.,   0.,   0.,   0.,   0.],
       [ 16., 103.,   0.,   0.,   0.,   0.,   0.],
       [ 16., 104.,   1.,   0.,   1.,   0.,   0.],
       [ 27., 100.,   1.,   0.,   1.,   0.,   0.],
       [ 84., 101.,   0.,   0.,   0.,   0.,   0.],
       [ 51., 104.,   1.,   0.,   1.,   0.,   0.],
       [ 69., 102.,   0.,   0.,   0.,   0.,   0.],
       [ 82., 102.,   0.,   0.,   0.,   0.,   1.],
       [ 69., 103.,   0.,   0.,

## With Column Transformer

In [26]:
from sklearn.compose import ColumnTransformer

In [29]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1_fever', SimpleImputer(), ['fever']),
        ("tnf2_gender_city", OneHotEncoder(drop='first'), ["gender", "city"]),
        ("tnf3_cough", OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough'])
    ], 
    remainder='passthrough'
)

In [30]:
transformer.fit_transform(X_train)

array([[101.,   0.,   0.,   0.,   1.,   0.,  81.],
       [100.,   0.,   0.,   1.,   0.,   0.,   5.],
       [100.,   0.,   0.,   1.,   0.,   0.,  19.],
       [100.,   1.,   1.,   0.,   0.,   0.,  27.],
       [103.,   0.,   1.,   0.,   0.,   0.,  73.],
       [103.,   1.,   0.,   1.,   0.,   1.,  70.],
       [102.,   0.,   1.,   0.,   0.,   0.,  49.],
       [101.,   0.,   0.,   1.,   0.,   1.,  51.],
       [101.,   0.,   1.,   0.,   0.,   0.,  64.],
       [101.,   0.,   0.,   1.,   0.,   0.,  83.],
       [ 98.,   0.,   0.,   0.,   1.,   0.,  65.],
       [104.,   0.,   0.,   0.,   0.,   0.,  18.],
       [103.,   0.,   0.,   0.,   0.,   0.,  16.],
       [104.,   1.,   0.,   1.,   0.,   0.,  16.],
       [100.,   1.,   0.,   1.,   0.,   0.,  27.],
       [101.,   0.,   0.,   0.,   0.,   0.,  84.],
       [104.,   1.,   0.,   1.,   0.,   0.,  51.],
       [102.,   0.,   0.,   0.,   0.,   0.,  69.],
       [102.,   0.,   0.,   0.,   0.,   1.,  82.],
       [103.,   0.,   0.,   1.,

In [31]:
transformer.transform(X_test).shape

(20, 7)